<a href="https://colab.research.google.com/github/aimedben/Ai-kabyle/blob/main/docs/OmniVoice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OmniVoice Quick Start

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/k2-fsa/OmniVoice/blob/master/docs/OmniVoice.ipynb)

This notebook demonstrates the basic usage of [OmniVoice](https://github.com/k2-fsa/OmniVoice), a massively multilingual zero-shot TTS model supporting 600+ languages.

**Contents:**
1. Installation
2. Option A — Gradio Demo (interactive web UI, no code needed)
3. Option B — Python API
   - 3.1 Load Model
   - 3.2 Voice Cloning
   - 3.3 Voice Design
   - 3.4 Auto Voice

## 1. Installation

Colab already provides a compatible PyTorch + CUDA environment, so we only need to install OmniVoice.

In [ ]:
!pip install omnivoice

## 2. Option A — Gradio Demo

Launch an interactive web UI with a public Gradio link. The `--share` flag creates a temporary public URL so you can access the demo from any browser.

> **If you prefer to use the Python API directly, skip to Option B below.**

In [ ]:
!omnivoice-demo --share

## 3. Option B — Python API

### 3.1 Load Model

In [ ]:
from omnivoice import OmniVoice
import soundfile as sf
import torch
from IPython.display import Audio, display

model = OmniVoice.from_pretrained(
    "k2-fsa/OmniVoice",
    device_map="cuda:0",
    dtype=torch.float16,
    load_asr=True,
)

### 3.2 Voice Cloning

Clone a voice from a short (3-10s) reference audio clip. Upload your own `ref.wav` or use any audio file.

`ref_text` is optional — if omitted, the model uses Whisper ASR to auto-transcribe it.

In [ ]:
from google.colab import files

print("Upload a reference audio file (wav/mp3/flac):")
uploaded = files.upload()
ref_audio_path = list(uploaded.keys())[0]
print(f"Uploaded: {ref_audio_path}")

In [ ]:
audio = model.generate(
    text="Hello, this is a test of zero-shot voice cloning.",
    ref_audio=ref_audio_path,
    # ref_text="Transcription of the reference audio.",  # optional
)

sf.write("clone_out.wav", audio[0], 24000)
display(Audio(audio[0], rate=24000))

### 3.3 Voice Design

Describe the desired voice with speaker attributes — no reference audio needed.

Supported attributes: gender, age, pitch, style (whisper), English accent, Chinese dialect. See [docs/voice-design.md](https://github.com/k2-fsa/OmniVoice/blob/master/docs/voice-design.md) for the full list.

In [ ]:
audio = model.generate(
    text="Hello, this is a test of zero-shot voice design.",
    instruct="female, low pitch, british accent",
)

sf.write("design_out.wav", audio[0], 24000)
display(Audio(audio[0], rate=24000))

### 3.4 Auto Voice

Let the model choose a voice automatically — no reference audio or instruct needed.

In [ ]:
audio = model.generate(
    text="This is a sentence generated with automatic voice selection.",
)

sf.write("auto_out.wav", audio[0], 24000)
display(Audio(audio[0], rate=24000))

In [1]:
import kagglehub
path = kagglehub.dataset_download("aimedbensa/common-voice-scripted-speech-27-0-kabyle")

Using Colab cache for faster access to the 'common-voice-scripted-speech-27-0-kabyle' dataset.


In [3]:
import os
import pandas as pd
from IPython.display import Audio, display

path = "/content/kabyle_data"

# 1. Trouver validated.tsv et charger les phrases
tsv_path = None
for root, dirs, files in os.walk(path):
    if "validated.tsv" in files:
        tsv_path = os.path.join(root, "validated.tsv")
        break

df = pd.read_csv(tsv_path, sep="\t", low_memory=False)
sentences_map = dict(zip(df["path"], df["sentence"]))

# 2. Récupérer 5 audios et s'arrêter immédiatement
exemples = []
for root, dirs, files in os.walk(path):
    for f in files:
        if f.endswith(".mp3") and f in sentences_map:
            exemples.append((os.path.join(root, f), sentences_map[f]))
            if len(exemples) == 10:
                break
    if len(exemples) == 10:
        break

# 3. Afficher les textes et lire les vocaux
for i, (audio_path, phrase) in enumerate(exemples, 1):
    print(f"--- Exemple {i} ---")
    print("Phrase (kabyle) :", phrase)
    display(Audio(audio_path))

--- Exemple 1 ---
Phrase (kabyle) : Yewwi-t-id cennegnagi!


--- Exemple 2 ---
Phrase (kabyle) : Tom yenɛel Mary.


--- Exemple 3 ---
Phrase (kabyle) : Zgiɣ qqaṛeɣ.


--- Exemple 4 ---
Phrase (kabyle) : Awal yeḥwağ asefru.


--- Exemple 5 ---
Phrase (kabyle) : Zzit-agi, am umeṭṭi uṛumi.


--- Exemple 6 ---
Phrase (kabyle) : Ay amcum deg-neɣ yeffɣen.


--- Exemple 7 ---
Phrase (kabyle) : Ad t-neǧǧ i uzekka.


--- Exemple 8 ---
Phrase (kabyle) : Ur gziɣ ara tamuɣli-ya-inem.


--- Exemple 9 ---
Phrase (kabyle) : Tayemmat d tameṭṭut yesɛan dderya.


--- Exemple 10 ---
Phrase (kabyle) : D tamacint usired yettεemmiren sufella.


In [4]:
!pip install -q transformers datasets accelerate peft evaluate jiwer librosa soundfile

In [5]:
import os
import pandas as pd
from datasets import Dataset, Audio

path = "/content/kabyle_data"

# 1. Charger validated.tsv
tsv_path = None
for root, dirs, files in os.walk(path):
    if "validated.tsv" in files:
        tsv_path = os.path.join(root, "validated.tsv")
        break

df = pd.read_csv(tsv_path, sep="\t", low_memory=False)
sentences_map = dict(zip(df["path"], df["sentence"]))

# 2. Indexer 5 000 audios validés et s'arrêter dès que le quota est atteint
rows = []
for root, dirs, files in os.walk(path):
    for f in files:
        if f.endswith(".mp3") and f in sentences_map:
            rows.append({"audio": os.path.join(root, f), "sentence": str(sentences_map[f])})
            if len(rows) >= 5000:
                break
    if len(rows) >= 5000:
        break

# 3. Créer le dataset Hugging Face (4 500 train / 500 validation) en 16 kHz
raw_ds = Dataset.from_pandas(pd.DataFrame(rows))
raw_ds = raw_ds.cast_column("audio", Audio(sampling_rate=16000))
dataset = raw_ds.train_test_split(test_size=0.1, seed=42)

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 4500
    })
    test: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 500
    })
})


In [2]:
import os
import torch
import pandas as pd
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from google.colab import drive
from datasets import Dataset, Audio
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
from peft import LoraConfig, get_peft_model

# 0. Monter Google Drive pour sauvegarder le modèle
drive.mount("/content/drive")
output_dir = "/content/drive/MyDrive/whisper-kabyle-lora"

# 1. Reconstruire rapidement le dataset depuis /content/kabyle_data (déjà extrait)
path = "/content/kabyle_data"
tsv_path = None
for root, dirs, files in os.walk(path):
    if "validated.tsv" in files:
        tsv_path = os.path.join(root, "validated.tsv")
        break

df = pd.read_csv(tsv_path, sep="\t", low_memory=False)
sentences_map = dict(zip(df["path"], df["sentence"]))

rows = []
for root, dirs, files in os.walk(path):
    for f in files:
        if f.endswith(".mp3") and f in sentences_map:
            rows.append({"audio": os.path.join(root, f), "sentence": str(sentences_map[f])})
            if len(rows) >= 5000:
                break
    if len(rows) >= 5000:
        break

raw_ds = Dataset.from_pandas(pd.DataFrame(rows))
raw_ds = raw_ds.cast_column("audio", Audio(sampling_rate=16000))
dataset = raw_ds.train_test_split(test_size=0.1, seed=42)

# 2. Charger Whisper-small et appliquer LoRA
model_name = "openai/whisper-small"
processor = WhisperProcessor.from_pretrained(model_name, language="French", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(model_name).to("cuda")

model.config.use_cache = False
model.generation_config.language = "french"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# 3. Transformation à la volée (évite de saturer les 12.67 Go de RAM)
def transform_batch(batch):
    input_features = [
        processor.feature_extractor(a["array"], sampling_rate=a["sampling_rate"]).input_features[0]
        for a in batch["audio"]
    ]
    labels = [processor.tokenizer(s).input_ids for s in batch["sentence"]]
    return {"input_features": input_features, "labels": labels}

dataset["train"].set_transform(transform_batch)
dataset["test"].set_transform(transform_batch)

# 4. Data Collator
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

# 5. Paramètres d'entraînement légers en RAM
training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-3,
    warmup_steps=100,
    max_steps=800,
    fp16=True,
    dataloader_num_workers=0, # Évite de dupliquer les processus en RAM
    eval_strategy="steps",
    eval_steps=200,
    save_steps=200,
    logging_steps=50,
    save_total_limit=2,
    remove_unused_columns=False,
    label_names=["labels"],
    report_to="none",
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
    processing_class=processor.feature_extractor,
)

# 6. Lancer l'entraînement et sauvegarder dans Google Drive
trainer.train()
model.save_pretrained(output_dir)
processor.save_pretrained(output_dir)
print("Modèle sauvegardé avec succès dans :", output_dir)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

trainable params: 1,769,472 || all params: 243,504,384 || trainable%: 0.7267


Step,Training Loss,Validation Loss
200,2.617972,1.284068
400,1.789581,1.037863
600,1.271211,0.929075
800,1.116769,0.889548


Modèle sauvegardé avec succès dans : /content/drive/MyDrive/whisper-kabyle-lora


In [8]:
!pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 87.2 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [3]:
!pip install -q peft librosa soundfile

In [5]:
import torch
import librosa
from google.colab import drive, files
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import PeftModel

# 1. Connecter Google Drive
drive.mount("/content/drive")
lora_dir = "/content/drive/MyDrive/whisper-kabyle-lora"

# 2. Charger Whisper-small + votre adaptateur entraîné
base_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
model = PeftModel.from_pretrained(base_model, lora_dir).to("cuda")
model.eval()

processor = WhisperProcessor.from_pretrained(lora_dir)

# 3. Importer un de vos fichiers vocaux (.mp3, .wav, .ogg, .m4a)
uploaded = files.upload()
audio_path = list(uploaded.keys())[0]

# 4. Transcrire en kabyle
audio, _ = librosa.load(audio_path, sr=16000)
inputs = processor.feature_extractor(audio, sampling_rate=16000, return_tensors="pt").input_features.to("cuda")

with torch.no_grad():
    predicted_ids = model.generate(inputs, language="french", task="transcribe")

transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
print("\n--- Transcription en kabyle ---")
print(transcription)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Saving Enregistrement (4).m4a to Enregistrement (4).m4a


/tmp/ipykernel_25874/651660601.py:23: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, _ = librosa.load(audio_path, sr=16000)
/usr/local/lib/python3.13/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)



--- Transcription en kabyle ---
Ad s ur lgism i wa iman, ttḥabaɛ fiɛzmanin, ttḥabaɛ tismaḍin.
